# Обзор результатов ML-эксперимента foF2

Выполненная версия обзорного ноутбука для проверки результатов 24-часового прогноза foF2.

Основные этапы:

1. Загружаются итоговые таблицы метрик.
2. Отбираются корректные строки результатов.
3. Сравниваются модели по RMSE, MAE, R² и корреляции.
4. Выделяются лучшие модели и станции с проблемными результатами.

Этот файл хранит уже выполненные ячейки и может использоваться как быстрый отчет по эксперименту.

In [1]:
# Подготовка библиотек, путей к данным и общих настроек ноутбука.
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

BASE = Path('..') if Path('../run_ml_baselines.py').exists() else Path('.')
RUN_DIR = BASE / 'artifacts' / 'fof2_fast_ml_24h'
STATION = 'EB040'   # None = first available station
HORIZON = '24h'     # None = all horizons
MODEL = 'CatBoost'  # None = all models
TRAIN_DAYS = 28     # None = all training windows
TOP_N = 20

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)
print('RUN_DIR =', RUN_DIR.resolve())


RUN_DIR = D:\IonoAutoML\artifacts\fof2_fast_ml_24h


In [2]:
# Функция безопасно читает таблицу результатов, если файл существует.
def read_table(path):
    path = Path(path)
    parquet = path.with_suffix('.parquet')
    if parquet.exists():
        try:
            return pd.read_parquet(parquet)
        except ImportError:
            pass
    if path.exists() and path.stat().st_size > 2:
        try:
            return pd.read_csv(path)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    return pd.DataFrame()

metrics = read_table(RUN_DIR / 'metrics.csv')
summary = read_table(RUN_DIR / 'metrics_summary.csv')
registry = read_table(RUN_DIR / 'model_registry.csv')
best_params = read_table(RUN_DIR / 'best_params.csv')
trials = read_table(RUN_DIR / 'optuna_trials.csv')
importance = read_table(RUN_DIR / 'feature_importance.csv')
station_summary = read_table(RUN_DIR / 'station_run_summary.csv')
manifest_path = RUN_DIR / 'run_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

print('metrics:', metrics.shape)
print('summary:', summary.shape)
print('registry:', registry.shape)
print('best_params:', best_params.shape)
print('trials:', trials.shape)
print('importance:', importance.shape)
print('station_summary:', station_summary.shape)
display(station_summary.head(20))


metrics: (5460, 24)
summary: (5, 9)
registry: (5460, 16)
best_params: (0, 0)
trials: (0, 0)
importance: (826235, 12)
station_summary: (1, 6)


,station,metrics_rows,prediction_rows,trial_rows,best_param_rows,feature_importance_rows
0,EB040,5460,524160,0,0,826235


## Фильтрация строк результатов

In [3]:
# Функция оставляет только корректные строки метрик для дальнейшего сравнения.
def apply_filters(df):
    if df.empty:
        return df
    out = df.copy()
    if STATION and 'station' in out.columns:
        out = out[out['station'] == STATION]
    if HORIZON and 'horizon' in out.columns:
        out = out[out['horizon'] == HORIZON]
    if MODEL and 'model' in out.columns:
        out = out[out['model'] == MODEL]
    if TRAIN_DAYS is not None and 'train_days' in out.columns:
        out = out[out['train_days'] == TRAIN_DAYS]
    return out

m = apply_filters(metrics)
r = apply_filters(registry)
bp = apply_filters(best_params)
tr = apply_filters(trials)
fi = apply_filters(importance)
print('filtered metrics:', m.shape)
display(m.head())


filtered metrics: (364, 24)


,latitude,latitude_zone,station,horizon,horizon_hours,split_id,train_days,model,train_rows,validation_rows,test_rows,feature_count,n,mae,rmse,r2,corr,bias,train_label_rows,train_start,train_end,safe_train_end,test_start,test_end
3644,40.8,N_mid,EB040,24h,24.0,2025-01-01_28d,28,CatBoost,2592,0,96,151,96,0.956082,1.244029,0.887121,0.972445,-0.050438,2582,2024-12-04T00:00:00+00:00,2024-12-31T23:45:00+00:00,2024-12-30T23:45:00+00:00,2025-01-01T00:00:00+00:00,2025-01-01T23:45:00+00:00
3649,40.8,N_mid,EB040,24h,24.0,2025-01-02_28d,28,CatBoost,2592,0,96,151,95,0.751450,1.035882,0.890959,0.958533,0.522131,2580,2024-12-05T00:00:00+00:00,2025-01-01T23:45:00+00:00,2024-12-31T23:45:00+00:00,2025-01-02T00:00:00+00:00,2025-01-02T23:45:00+00:00
3654,40.8,N_mid,EB040,24h,24.0,2025-01-03_28d,28,CatBoost,2592,0,96,151,96,0.904055,1.061813,0.886545,0.945405,-0.241551,2580,2024-12-06T00:00:00+00:00,2025-01-02T23:45:00+00:00,2025-01-01T23:45:00+00:00,2025-01-03T00:00:00+00:00,2025-01-03T23:45:00+00:00
3659,40.8,N_mid,EB040,24h,24.0,2025-01-04_28d,28,CatBoost,2592,0,96,151,96,0.567632,0.795890,0.939695,0.971219,-0.021305,2580,2024-12-07T00:00:00+00:00,2025-01-03T23:45:00+00:00,2025-01-02T23:45:00+00:00,2025-01-04T00:00:00+00:00,2025-01-04T23:45:00+00:00
3664,40.8,N_mid,EB040,24h,24.0,2025-01-05_28d,28,CatBoost,2592,0,96,151,95,0.643517,0.832486,0.939394,0.983704,-0.316796,2581,2024-12-08T00:00:00+00:00,2025-01-04T23:45:00+00:00,2025-01-03T23:45:00+00:00,2025-01-05T00:00:00+00:00,2025-01-05T23:45:00+00:00


## Краткая сводка результатов

In [4]:
# Формируется краткая сводка по доступным результатам моделей.
if not m.empty:
    metric_cols = [c for c in ['mae', 'rmse', 'r2', 'corr', 'bias'] if c in m.columns]
    by_window_model = (
        m.groupby(['station', 'horizon', 'train_days', 'model'], dropna=False)[metric_cols]
        .mean(numeric_only=True)
        .reset_index()
        .sort_values(['mae', 'rmse'])
    )
    display(by_window_model.head(TOP_N))

    best_by_window = (
        by_window_model.sort_values(['station', 'horizon', 'train_days', 'mae', 'rmse'])
        .groupby(['station', 'horizon', 'train_days'], dropna=False)
        .head(1)
    )
    display(best_by_window)
else:
    print('No metrics after filters')


,station,horizon,train_days,model,mae,rmse,r2,corr,bias
0,EB040,24h,28,CatBoost,0.736747,0.904887,0.60664,0.885259,-0.001089


,station,horizon,train_days,model,mae,rmse,r2,corr,bias
0,EB040,24h,28,CatBoost,0.736747,0.904887,0.60664,0.885259,-0.001089


## Лучшие модели по RMSE

In [5]:
# Формируется краткая сводка по доступным результатам моделей.
if not m.empty:
    best = (
        m.sort_values('rmse')
        .groupby(['station', 'horizon', 'train_days'], dropna=False)
        .head(1)
        .sort_values(['station', 'horizon', 'train_days'])
    )
    cols = [c for c in ['station','latitude_zone','horizon','train_days','split_id','model','n','mae','rmse','r2','corr','bias','best_val_score'] if c in best.columns]
    display(best[cols].head(TOP_N))
else:
    print('Нет metrics. Сначала запусти run_ml_baselines.py.')


,station,latitude_zone,horizon,train_days,split_id,model,n,mae,rmse,r2,corr,bias
4729,EB040,N_mid,24h,28,2025-08-06_28d,CatBoost,95,0.27754,0.347066,0.745764,0.901744,0.094734


In [6]:
# Формируется краткая сводка по доступным результатам моделей.
if not m.empty:
    metric_cols = [c for c in ['rmse','mae','r2','corr','bias'] if c in m.columns]
    agg = m.groupby(['horizon','model'], dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
    fig = px.bar(agg, x='model', y='rmse', color='horizon', barmode='group', title='Mean RMSE by model and horizon', height=520)
    fig.update_xaxes(tickangle=35)
    fig.show()


## Широтные зоны

In [7]:
# Формируется краткая сводка по доступным результатам моделей.
if not m.empty and 'latitude_zone' in m.columns:
    metric_cols = [c for c in ['rmse','mae','r2','corr','bias'] if c in m.columns]
    zone = m.groupby(['latitude_zone','horizon','model'], dropna=False)[metric_cols].mean(numeric_only=True).reset_index()
    display(zone.sort_values(['latitude_zone','horizon','rmse']).head(TOP_N))
    fig = px.bar(zone, x='model', y='rmse', color='latitude_zone', facet_col='horizon', barmode='group', title='RMSE by latitude zone', height=520)
    fig.update_xaxes(tickangle=35)
    fig.show()
else:
    print('latitude_zone пока нет в metrics или metrics пустой')


,latitude_zone,horizon,model,rmse,mae,r2,corr,bias
0,N_mid,24h,CatBoost,0.904887,0.736747,0.60664,0.885259,-0.001089


## Прогнозы станции

In [8]:
# Функция `load_station_predictions` используется как вспомогательный шаг в расчетах ноутбука.
def load_station_predictions(station=None):
    stations_dir = RUN_DIR / 'stations'
    if not stations_dir.exists():
        return pd.DataFrame()
    if station is None:
        station_dirs = sorted(p for p in stations_dir.iterdir() if p.is_dir())
        if not station_dirs:
            return pd.DataFrame()
        station_dir = station_dirs[0]
    else:
        station_dir = stations_dir / station
    parquet = station_dir / 'predictions.parquet'
    csv = station_dir / 'predictions.csv'
    if parquet.exists():
        try:
            return pd.read_parquet(parquet)
        except ImportError:
            pass
    if csv.exists():
        return pd.read_csv(csv, parse_dates=['time_utc','target_time_utc'])
    return pd.DataFrame()

pred = load_station_predictions(STATION)
if not pred.empty:
    pred['time_utc'] = pd.to_datetime(pred['time_utc'], utc=True, errors='coerce')
    pred['target_time_utc'] = pd.to_datetime(pred['target_time_utc'], utc=True, errors='coerce')
    pred = apply_filters(pred)
print('predictions:', pred.shape)
display(pred.head())


predictions: (34944, 9)


,station,time_utc,target_time_utc,horizon,split_id,train_days,model,actual,predicted
349824,EB040,2025-01-01 00:00:00+00:00,2025-01-02 00:00:00+00:00,24h,2025-01-01_28d,28,CatBoost,2.133333,3.992287
349825,EB040,2025-01-01 00:15:00+00:00,2025-01-02 00:15:00+00:00,24h,2025-01-01_28d,28,CatBoost,2.766667,4.011367
349826,EB040,2025-01-01 00:30:00+00:00,2025-01-02 00:30:00+00:00,24h,2025-01-01_28d,28,CatBoost,2.933333,4.008289
349827,EB040,2025-01-01 00:45:00+00:00,2025-01-02 00:45:00+00:00,24h,2025-01-01_28d,28,CatBoost,3.016667,4.054281
349828,EB040,2025-01-01 01:00:00+00:00,2025-01-02 01:00:00+00:00,24h,2025-01-01_28d,28,CatBoost,3.083333,4.139167


In [9]:
# Выполнение расчетного шага и вывод промежуточного результата.
if not pred.empty:
    plot_station = STATION or pred['station'].iloc[0]
    plot_horizon = HORIZON or pred['horizon'].iloc[0]
    plot_model = MODEL or pred['model'].iloc[0]
    plot_train_days = TRAIN_DAYS or pred['train_days'].iloc[0]
    subset = pred[
        (pred['station'] == plot_station)
        & (pred['horizon'] == plot_horizon)
        & (pred['model'] == plot_model)
        & (pred['train_days'] == plot_train_days)
    ].copy()
    subset = subset.sort_values('target_time_utc').drop_duplicates(['target_time_utc', 'model', 'train_days'])
    subset = subset.head(96 * 14)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=subset['target_time_utc'], y=subset['actual'], mode='lines', name='actual'))
    fig.add_trace(go.Scatter(x=subset['target_time_utc'], y=subset['predicted'], mode='lines', name='predicted'))
    fig.update_layout(
        title=f'{plot_station} {plot_horizon} {plot_model} window={plot_train_days}d: actual vs predicted',
        xaxis_title='Target time UTC',
        yaxis_title='foF2 MHz',
        height=520,
    )
    fig.show()
else:
    print('??? predictions ??? ?????????')


## Best Params ? Trials

In [10]:
# Выполнение расчетного шага и вывод промежуточного результата.
if not bp.empty:
    cols = [c for c in ['station','latitude_zone','horizon','train_days','split_id','model','search_method','best_value','best_params'] if c in bp.columns]
    bp_view = bp.loc[:, cols].copy()
    sort_cols = [c for c in ['station','horizon','best_value'] if c in bp_view.columns]
    if sort_cols:
        bp_view = bp_view.sort_values(by=sort_cols)
    display(bp_view.head(TOP_N))
else:
    print('best_params пустой')

if not tr.empty:
    cols = [c for c in ['station','latitude_zone','horizon','train_days','split_id','model','trial_number','value','state','params'] if c in tr.columns]
    tr_view = tr.loc[:, cols].copy()
    if 'value' in tr_view.columns:
        tr_view = tr_view.sort_values(by='value')
    display(tr_view.head(TOP_N))
else:
    print('trials пустой')


best_params пустой
trials пустой


## Важность признаков

In [11]:
# Выполнение расчетного шага и вывод промежуточного результата.
if not fi.empty:
    top_features = (
        fi.groupby(['model','feature'], dropna=False)['importance_abs']
        .mean()
        .reset_index()
        .sort_values('importance_abs', ascending=False)
        .head(TOP_N)
    )
    display(top_features)
    fig = px.bar(top_features.sort_values('importance_abs'), x='importance_abs', y='feature', color='model', orientation='h', title='Top feature importance / coefficients', height=650)
    fig.show()
else:
    print('feature_importance пустой')


,model,feature,importance_abs
73,CatBoost,foF2p,8.581057
136,CatBoost,hour_cos,5.044131
1,CatBoost,MUF_D,2.412662
10,CatBoost,MUF_D_lag_7D,2.279668
16,CatBoost,TEC_lag_24h,2.274211
19,CatBoost,TEC_lag_48h,2.140329
5,CatBoost,MUF_D_lag_24h,1.960835
17,CatBoost,TEC_lag_30min,1.839603
72,CatBoost,foF2_state,1.832225
3,CatBoost,MUF_D_lag_15min,1.830378


## Manifest

In [12]:
# Выполнение расчетного шага и вывод промежуточного результата.
if manifest:
    print(json.dumps({k: manifest.get(k) for k in ['stations','horizons','metrics_rows','prediction_rows','trial_rows','best_param_rows','feature_importance_rows','result_layout']}, ensure_ascii=False, indent=2))
else:
    print('run_manifest.json пока нет')


{
  "stations": [
    "EB040"
  ],
  "horizons": [
    "24h"
  ],
  "metrics_rows": 5460,
  "prediction_rows": 524160,
  "trial_rows": 0,
  "best_param_rows": 0,
  "feature_importance_rows": 826235,
  "result_layout": {
    "global": [
      "metrics.csv",
      "metrics_summary.csv",
      "optuna_trials.csv",
      "best_params.csv",
      "feature_importance.csv",
      "model_registry.csv",
      "experiment_report.md",
      "run_manifest.json"
    ],
    "per_station": "artifacts/<experiment>/stations/<station>/{metrics,predictions,optuna_trials,best_params,feature_importance}.{csv,parquet}",
    "models": "artifacts/<experiment>/models/<station>_<horizon>_<model>_<split_id>.joblib",
    "optuna_storage": "sqlite:///artifacts/fof2_fast_ml_24h/optuna.db"
  }
}
